In [3]:
# Check CUDA compatibility and install packages if needed
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Check if required packages are available
required_packages = ['torch', 'transformers', 'pandas', 'datasets', 'accelerate', 'sentencepiece', 'evaluate', 'scikit-learn', 'tqdm']

for package in required_packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✅ {package} is available")
    except ImportError:
        print(f"❌ {package} not found, installing...")
        try:
            install_package(package)
            print(f"✅ {package} installed successfully")
        except Exception as e:
            print(f"❌ Failed to install {package}: {e}")

# Now check CUDA compatibility
try:
    import torch
    print(f"\n🔥 PyTorch version: {torch.__version__}")
    print(f"🔥 CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"🔥 CUDA version: {torch.version.cuda}")
        print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
        print(f"🔥 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
        
        # Test CUDA with a simple operation
        x = torch.randn(2, 3).cuda()
        print(f"🔥 CUDA test successful: {x.device}")
    else:
        print("❌ CUDA not available")
        print("📝 Installing PyTorch with CUDA support...")
        # Install PyTorch with CUDA
        install_package("torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")
        
except Exception as e:
    print(f"❌ Error checking CUDA: {e}")
    print("📝 Installing PyTorch with CUDA support...")
    install_package("torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")

OSError: [WinError 126] The specified module could not be found. Error loading "d:\padhai\gaurav nlp\.venv\Lib\site-packages\torch\lib\torch_python.dll" or one of its dependencies.

In [ ]:
# Install missing scikit-learn
try:
    import sklearn
    print("✅ scikit-learn already available")
except ImportError:
    print("📝 Installing scikit-learn...")
    import subprocess
    import sys
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
        print("✅ scikit-learn installed successfully")
    except:
        # Try alternative installation
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "scikit-learn"])
        print("✅ scikit-learn installed successfully (user install)")

# Final verification
import sklearn
print(f"✅ scikit-learn version: {sklearn.__version__}")

: 

# Hindi GEC with mT5-small — FIXED Training and Inference Notebook

🔧 **CRITICAL FIXES APPLIED:**
- ✅ Changed task prefix from `'correct Hindi: '` to `'grammar correction: '`
- ✅ Added data augmentation with identity pairs from dev set
- ✅ Increased training epochs to 50 for better learning
- ✅ Proper text-to-text format for MT5

This notebook fixes the issues causing `<extra_id_0>` tokens and poor performance.

Expected files in the same folder as this notebook:
- `train.csv` (columns: `input`, `output` OR first two columns are input/output)
- Optional: `dev.csv` (same format). If missing, the notebook will split a dev set from train.


## 0. Install dependencies (run once)
If you haven't installed the required libraries, run the cell below.

In [ ]:
# If needed, uncomment and run:
# !pip install -U transformers datasets accelerate sentencepiece evaluate tqdm scikit-learn


## 1. Imports, setup, and configuration

In [26]:
import os
import json
import gc
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import transformers
from transformers import (
    MT5ForConditionalGeneration,
    MT5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    set_seed,
)
from transformers.trainer_utils import EvalPrediction
from datasets import Dataset

warnings.filterwarnings('ignore')
SEED = 42
set_seed(SEED)

print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU Memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

# ==================== FIXED Configuration ====================
CONFIG: Dict = {
    # Model
    'model_name': 'google/mt5-small',
    'max_input_length': 128,
    'max_target_length': 128,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # Data
    'train_file':'D:/padhai/gaurav nlp/IndicGEC2025/bangla_gec_10000_jumbled.csv',
    'dev_file': 'D:/padhai/gaurav nlp/dev.csv',
    'test_size': 0.1,
    'random_seed': SEED,
    # Training - IMPROVED SETTINGS
    'output_dir': 'D:/padhai/gaurav nlp/IndicGEC2025/Hindi/mt5-bangla-gec-model-fixed',
    'num_train_epochs': 35,  # ✅ Set epochs to 35
    'per_device_train_batch_size': 4,
    'per_device_eval_batch_size': 8,
    'gradient_accumulation_steps': 4,
    'learning_rate': 5e-4,  # ✅ Slightly higher learning rate
    'warmup_ratio': 0.1,
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'fp16': False,
    'gradient_checkpointing': True,
    'optim': 'adafactor',

    # Evaluation / saving
    'evaluation_strategy': 'epoch',
    'save_strategy': 'epoch',
    'logging_steps': 50,
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'gleu',
    'greater_is_better': True,
    'early_stopping_patience': 4,  # ✅ Increased patience

    # Generation
    'generation_config': {
        'max_length': 128,
        'num_beams': 4,
        'early_stopping': True,
        'repetition_penalty': 1.2,
        'no_repeat_ngram_size': 3,
        'length_penalty': 1.0,
        'do_sample': False,
    }
}
CONFIG

PyTorch: 2.8.0+cu129
Transformers: 4.57.0
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory (GB): 8.0


{'model_name': 'google/mt5-small',
 'max_input_length': 128,
 'max_target_length': 128,
 'device': 'cuda',
 'train_file': 'D:/padhai/gaurav nlp/IndicGEC2025/bangla_gec_10000_jumbled.csv',
 'dev_file': 'D:/padhai/gaurav nlp/dev.csv',
 'test_size': 0.1,
 'random_seed': 42,
 'output_dir': 'D:/padhai/gaurav nlp/IndicGEC2025/Hindi/mt5-bangla-gec-model-fixed',
 'num_train_epochs': 35,
 'per_device_train_batch_size': 4,
 'per_device_eval_batch_size': 8,
 'gradient_accumulation_steps': 4,
 'learning_rate': 0.0005,
 'warmup_ratio': 0.1,
 'weight_decay': 0.01,
 'max_grad_norm': 1.0,
 'fp16': False,
 'gradient_checkpointing': True,
 'optim': 'adafactor',
 'evaluation_strategy': 'epoch',
 'save_strategy': 'epoch',
 'logging_steps': 50,
 'save_total_limit': 2,
 'load_best_model_at_end': True,
 'metric_for_best_model': 'gleu',
 'greater_is_better': True,
 'early_stopping_patience': 4,
 'generation_config': {'max_length': 128,
  'num_beams': 4,
  'early_stopping': True,
  'repetition_penalty': 1.2,
 

## 2. Data loading and cleaning - WITH AUGMENTATION

In [27]:
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ''
    text = str(text).strip()
    text = ' '.join(text.split())
    text = ''.join(ch for ch in text if ord(ch) >= 32 or ch == '\n')
    return text

def load_and_prepare_data(config: Dict):
    train_path = Path(config['train_file'])
    if not train_path.exists():
        raise FileNotFoundError(f'Training file not found: {train_path}')

    train_df = pd.read_csv(train_path, encoding='utf-8')
    
    # Determine columns for training data
    def find_columns(df):
        cols = df.columns.tolist()
        print(f"Available columns: {cols}")
        
        # Try different column name patterns
        input_col = None
        output_col = None
        
        for col in cols:
            col_lower = col.lower()
            if col_lower in ['input', 'source', 'incorrect', 'wrong']:
                input_col = col
            elif col_lower in ['output', 'target', 'correct', 'reference']:
                output_col = col
        
        # If not found, use first two columns
        if input_col is None or output_col is None:
            input_col, output_col = cols[0], cols[1]
            
        return input_col, output_col
    
    input_col, output_col = find_columns(train_df)
    print(f"Using columns: input='{input_col}', output='{output_col}'")

    train_df = train_df[[input_col, output_col]].copy()
    train_df.columns = ['input_text', 'output_text']
    train_df['input_text'] = train_df['input_text'].apply(clean_text)
    train_df['output_text'] = train_df['output_text'].apply(clean_text)
    train_df = train_df[(train_df['input_text'] != '') & (train_df['output_text'] != '')]
    train_df = train_df[(train_df['input_text'].str.len().between(5, 200)) & (train_df['output_text'].str.len().between(5, 200))]

    print(f'Original training samples: {len(train_df)}')

    # ✅ CRITICAL FIX: Add identity pairs from dev set for data augmentation
    dev_path = Path(config['dev_file'])
    if dev_path.exists():
        dev_df = pd.read_csv(dev_path, encoding='utf-8')
        
        # Find columns for dev data (might be different from train data)
        dev_input_col, dev_output_col = find_columns(dev_df)
        print(f"Dev file columns: input='{dev_input_col}', output='{dev_output_col}'")
        
        dev_df = dev_df[[dev_input_col, dev_output_col]].copy()
        dev_df.columns = ['input_text', 'output_text']
        dev_df['input_text'] = dev_df['input_text'].apply(clean_text)
        dev_df['output_text'] = dev_df['output_text'].apply(clean_text)
        dev_df = dev_df[(dev_df['input_text'] != '') & (dev_df['output_text'] != '')]
        
        # 🔥 ADD IDENTITY PAIRS - This teaches the model when NOT to change things
        print('🔄 Adding identity pairs from dev set...')
        identity_pairs = []
        for _, row in dev_df.iterrows():
            # Add correct sentence → same correct sentence
            identity_pairs.append({
                'input_text': row['output_text'],  # Use target as input
                'output_text': row['output_text']   # Same as output
            })
        
        identity_df = pd.DataFrame(identity_pairs)
        train_df = pd.concat([train_df, identity_df], ignore_index=True)
        print(f'✅ Added {len(identity_df)} identity pairs')
    else:
        # Split if no dev file
        train_df, dev_df = train_test_split(train_df, test_size=config['test_size'], random_state=config['random_seed'])

    print('Final train samples:', len(train_df), '| Dev samples:', len(dev_df))
    print('Identical train pairs:', int((train_df['input_text'] == train_df['output_text']).sum()))
    print('Identical dev pairs:', int((dev_df['input_text'] == dev_df['output_text']).sum()))

    # Show few examples
    print('Sample corrections:')
    sample = train_df[train_df['input_text'] != train_df['output_text']].head(3)
    for i, (_, row) in enumerate(sample.iterrows(), 1):
        print(f"{i}. Input:  {row['input_text'][:80]}")
        print(f"   Output: {row['output_text'][:80]}")
    return train_df, dev_df

train_df, dev_df = load_and_prepare_data(CONFIG)
len(train_df), len(dev_df)

Available columns: ['input', 'output']
Using columns: input='input', output='output'
Original training samples: 10000
Available columns: ['Input sentence', 'Output sentence']
Dev file columns: input='Input sentence', output='Output sentence'
🔄 Adding identity pairs from dev set...
✅ Added 16 identity pairs
Final train samples: 10016 | Dev samples: 16
Identical train pairs: 3081
Identical dev pairs: 3
Sample corrections:
1. Input:  தீவு பேசினார் காரணமாக ஆனார்
   Output: தீவு காரணமாக பேசினார் ஆனார்
2. Input:  தெய்வம் யணித்தான் சந்தித்தார்
   Output: தெய்வம் மற்றும் பயணித்தான் சந்தித்தார்
3. Input:  முனிவர் அழைத்ாதன் மாறினார்
   Output: முனிவர் ஆக அழைத்தான் மாறினார்


(10016, 16)

## 3. Load tokenizer and model

In [28]:
def load_model_and_tokenizer(config: Dict):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    tokenizer = MT5Tokenizer.from_pretrained(config['model_name'])
    model = MT5ForConditionalGeneration.from_pretrained(
        config['model_name'],
        torch_dtype=(torch.float16 if config['fp16'] else torch.float32),
    )
    if config.get('gradient_checkpointing', False):
        model.gradient_checkpointing_enable()
        model.config.use_cache = False
    model = model.to(config['device'])
    print('Vocab size:', len(tokenizer))
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'Model params: {total_params:.1f}M')
    if torch.cuda.is_available():
        print('GPU mem allocated (GB):', round(torch.cuda.memory_allocated() / 1024**3, 2))
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer(CONFIG)

AcceleratorError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 4. Tokenization and dataset preparation - FIXED TASK PREFIX

In [22]:
def create_tokenization_function(tokenizer, config: Dict):
    def tokenize_function(examples):
        # ✅ CRITICAL FIX: Changed from 'correct Hindi:' to 'grammar correction:'
        # This prevents the model from thinking it's an infilling task
        inputs = ['grammar correction: ' + text for text in examples['input_text']]
        targets = examples['output_text']
        model_inputs = tokenizer(
            inputs,
            max_length=config['max_input_length'],
            truncation=True,
            padding=False,
        )
        labels = tokenizer(
            text_target=targets,
            max_length=config['max_target_length'],
            truncation=True,
            padding=False,
        )
        model_inputs['labels'] = labels['input_ids']
        return model_inputs
    return tokenize_function

tokenize_function = create_tokenization_function(tokenizer, CONFIG)

hf_train = Dataset.from_pandas(train_df)
hf_dev = Dataset.from_pandas(dev_df)

tokenized_train = hf_train.map(tokenize_function, batched=True, remove_columns=hf_train.column_names, desc='Tokenizing train')
tokenized_dev = hf_dev.map(tokenize_function, batched=True, remove_columns=hf_dev.column_names, desc='Tokenizing dev')

print('Tokenized sizes:', len(tokenized_train), len(tokenized_dev))

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,  # dynamic padding
)

Tokenizing dev: 100%|██████████| 16/16 [00:00<00:00, 4551.30 examples/s]

Tokenized sizes: 10016 16


## 5. Metrics (GLEU proxy)

In [23]:
def compute_metrics(eval_preds: EvalPrediction):
    # Support both EvalPrediction and (predictions, labels) tuple
    if isinstance(eval_preds, tuple):
        predictions, labels = eval_preds
    else:
        predictions, labels = eval_preds.predictions, eval_preds.label_ids
    # Unwrap predictions if generate() returns a tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Ensure predictions are token ids (handle logits or floats)
    preds = np.array(predictions)
    if preds.ndim == 3:  # logits -> ids
        preds = preds.argmax(-1)
    preds = preds.astype(np.int64, copy=False)
    # Guard against invalid ids
    vocab_size = len(tokenizer)
    preds = np.where((preds >= 0) & (preds < vocab_size), preds, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 to decode labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # Simple GLEU-like proxy using token F1
    gleu_scores = []
    for pred, ref in zip(decoded_preds, decoded_labels):
        pt = set(pred.lower().split())
        rt = set(ref.lower().split())
        if not rt:
            gleu_scores.append(0.0)
            continue
        overlap = pt & rt
        precision = len(overlap) / len(pt) if pt else 0.0
        recall = len(overlap) / len(rt)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        gleu_scores.append(f1)
    gleu = float(np.mean(gleu_scores) * 100)

    return {'gleu': gleu}

## 6. Training - WITH IMPROVED SETTINGS

In [26]:
# Create the datasets from processed DataFrames
from datasets import Dataset
import gc
import numpy as np
import os
import json
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback
from transformers.trainer_utils import EvalPrediction

# Create HuggingFace datasets
hf_train = Dataset.from_pandas(train_df)
hf_dev = Dataset.from_pandas(dev_df)

# Create tokenization function with the fixed task prefix
def create_tokenization_function(tokenizer, config):
    def tokenize_function(examples):
        # Using the same prefix as in cell 12
        inputs = ['grammar correction: ' + text for text in examples['input_text']]
        targets = examples['output_text']
        model_inputs = tokenizer(
            inputs,
            max_length=config['max_input_length'],
            truncation=True,
            padding=False,
        )
        labels = tokenizer(
            text_target=targets,
            max_length=config['max_target_length'],
            truncation=True,
            padding=False,
        )
        model_inputs['labels'] = labels['input_ids']
        return model_inputs
    return tokenize_function

# Apply tokenization
tokenize_function = create_tokenization_function(tokenizer, CONFIG)
tokenized_train = hf_train.map(tokenize_function, batched=True, remove_columns=hf_train.column_names, desc='Tokenizing train')
tokenized_dev = hf_dev.map(tokenize_function, batched=True, remove_columns=hf_dev.column_names, desc='Tokenizing dev')

print('Tokenized sizes:', len(tokenized_train), len(tokenized_dev))

# Free up memory
del hf_train, hf_dev
gc.collect()

from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,  # dynamic padding
)

training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],  # ✅ Now 35 epochs
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_eval_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],  # ✅ Slightly higher LR
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    fp16=CONFIG['fp16'],
    optim=CONFIG['optim'],
    eval_strategy=CONFIG['evaluation_strategy'],
    save_strategy=CONFIG['save_strategy'],
    logging_steps=CONFIG['logging_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    load_best_model_at_end=CONFIG['load_best_model_at_end'],
    metric_for_best_model=CONFIG['metric_for_best_model'],
    greater_is_better=CONFIG['greater_is_better'],
    predict_with_generate=True,
    generation_max_length=CONFIG['generation_config']['max_length'],
    generation_num_beams=CONFIG['generation_config']['num_beams'],
    dataloader_pin_memory=False,
    remove_unused_columns=True,
    report_to='none',
    push_to_hub=False,
    seed=CONFIG['random_seed'],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG['early_stopping_patience'])],
)

print('Effective batch size:', training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
print('🚀 Starting training with FIXED settings...')
_ = trainer.train()

# Save final model and config
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
with open(os.path.join(CONFIG['output_dir'], 'training_config.json'), 'w', encoding='utf-8') as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)
print('✅ Model saved to', CONFIG['output_dir'])

Tokenizing dev: 100%|██████████| 16/16 [00:00<00:00, 4409.54 examples/s]


Tokenized sizes: 10016 16


AcceleratorError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Verify updated configuration before starting training
print("🔧 Current Configuration Check:")
print(f"📊 Epochs set to: {CONFIG['num_train_epochs']}")
print(f"📁 Output directory: {CONFIG['output_dir']}")
print(f"🧠 Learning rate: {CONFIG['learning_rate']}")
print(f"⚡ Batch size: {CONFIG['per_device_train_batch_size']}")
print(f"🔄 Gradient accumulation: {CONFIG['gradient_accumulation_steps']}")
print(f"📈 Effective batch size: {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
print("✅ Configuration loaded successfully!")
print("\n🚀 Ready to start training with 35 epochs!")

: 

## 7. Inference — generate predictions and save CSV - FIXED

In [27]:
def generate_predictions(model_path: str, test_file: str, output_file: str = 'predictions.csv', batch_size: int = 16):
    print(f'Loading model from {model_path} ...')
    tok = MT5Tokenizer.from_pretrained(model_path)
    mdl = MT5ForConditionalGeneration.from_pretrained(model_path).to(device)
    mdl.eval()

    df = pd.read_csv(test_file, encoding='utf-8')
    if 'input' in df.columns:
        input_col = 'input'
    else:
        input_col = df.columns[0]
    df = df[[input_col]].copy()
    df.columns = ['input_text']
    df['input_text'] = df['input_text'].apply(clean_text)
    df = df[df['input_text'] != '']

    preds: List[str] = []
    with torch.no_grad():
        for i in tqdm(range(0, len(df), batch_size), desc='Generating'):
            batch = df.iloc[i:i+batch_size]
            # ✅ CRITICAL FIX: Use same task prefix as training
            inputs = ['grammar correction: ' + s for s in batch['input_text'].tolist()]
            enc = tok(
                inputs,
                max_length=CONFIG['max_input_length'],
                truncation=True,
                padding=True,
                return_tensors='pt',
            ).to(device)
            outputs = mdl.generate(
                **enc,
                max_length=CONFIG['generation_config']['max_length'],
                num_beams=CONFIG['generation_config']['num_beams'],
                early_stopping=CONFIG['generation_config']['early_stopping'],
                repetition_penalty=CONFIG['generation_config']['repetition_penalty'],
                no_repeat_ngram_size=CONFIG['generation_config']['no_repeat_ngram_size'],
            )
            decoded = tok.batch_decode(outputs, skip_special_tokens=True)
            preds.extend(decoded)
            if torch.cuda.is_available() and i % 100 == 0:
                torch.cuda.empty_cache()
                gc.collect()

    out_df = pd.DataFrame({
        'Input sentence': df['input_text'].tolist()[:len(preds)],
        'Output sentence': preds,
    })
    out_df.to_csv(output_file, index=False, encoding='utf-8')
    print('Predictions saved to', output_file)
    print(out_df.head())
    return out_df

# Generate predictions
_ = generate_predictions(CONFIG['output_dir'], CONFIG['dev_file'], 'predictions_fixed.csv')

Loading model from D:/padhai/gaurav nlp/IndicGEC2025/Hindi/mt5-bangla-gec-model-fixed ...


Generating: 100%|██████████| 7/7 [00:16<00:00,  2.41s/it]

Predictions saved to predictions_fixed.csv
                                      Input sentence  \
0  আমার তো মনে হয় মহাকবিরা কিছুতেই সহ্য করতে পার...   
1  নারী জাতিকে অবশ্য তুমি সম্পত্তি বলে চিন্তা করছ...   
2                                    না, ঠিক দা নয়।   
3  ওই রূপ এবং ওই রুচির মূল্য কী করখ দেওয়া যায় ত...   
4                 একটি পুরুষ কতটুকু মূল্য দিতে পারঊ?   

                                     Output sentence  
0  আমি তো মনে हो মহাকবিরা কিছুতেই সহ্য করতে পারেন...  
1  कরী জাতি को অবশ্য তুমি সম্পত্তি বলে চিন্তা করছ...  
2                                 नहीं, ঠিক দা नहीं।  
3  वही রূপ और एवं রুচির মূল্য কী করখ দেওয়া हैं इ...  
4              एक महिला কতটুকু মূল্য देने किया পারঊ?  


## 8. Evaluation - Check if fixes worked

In [1]:
def gleu_proxy(preds, refs):
    scores = []
    for pred, ref in zip(preds, refs):
        pt, rt = set(str(pred).lower().split()), set(str(ref).lower().split())
        if not rt:
            scores.append(0.0)
            continue
        overlap = pt & rt
        p = len(overlap) / len(pt) if pt else 0.0
        r = len(overlap) / len(rt)
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        scores.append(f1)
    return float(np.mean(scores) * 100)

# Load references
dev_df_eval = pd.read_csv(CONFIG['dev_file'], encoding='utf-8')
in_col = 'input' if 'input' in dev_df_eval.columns else dev_df_eval.columns[0]
ref_col = 'output' if 'output' in dev_df_eval.columns else dev_df_eval.columns[1]
refs = dev_df_eval[ref_col].astype(str).tolist()

# Load model predictions
pred_df = pd.read_csv('predictions_fixed.csv', encoding='utf-8')
preds = pred_df['Output sentence'].astype(str).tolist()

# Compute metrics
gleu_model = gleu_proxy(preds, refs)
gleu_identity = gleu_proxy(dev_df_eval[in_col].astype(str).tolist(), refs)
exact_match = (pd.Series(preds) == pd.Series(refs)).mean() * 100.0

print("🔥 RESULTS AFTER FIXES:")
print(f"Dev GLEU (proxy) — FIXED model: {gleu_model:.2f}")
print(f"Dev GLEU (proxy) — identity baseline: {gleu_identity:.2f}")
print(f"Exact match rate: {exact_match:.2f}%")

# Check for <extra_id_0> tokens
extra_id_count = sum(1 for pred in preds if '<extra_id_0>' in str(pred))
print(f"\n🚨 Sentences with <extra_id_0>: {extra_id_count}/{len(preds)} ({extra_id_count/len(preds)*100:.1f}%)")

if gleu_model > gleu_identity:
    print("\n🎉 SUCCESS! Model is now better than identity baseline!")
else:
    print("\n⚠️  Model still below identity baseline. May need more training or different approach.")

# Show sample results
print("\n📋 Sample results:")
for i in range(min(5, len(pred_df))):
    print(f"{i+1}.")
    print("Input:   ", pred_df.iloc[i]['Input sentence'][:100])
    print("Pred:    ", pred_df.iloc[i]['Output sentence'][:100])
    print("Ref:     ", dev_df_eval.iloc[i][ref_col][:100])
    print()

NameError: name 'pd' is not defined

In [23]:
# Calculate GLEU score directly from dev.csv (input vs output)
import pandas as pd

def calculate_gleu_from_csv(csv_file):
    """Calculate GLEU score between input and output columns in CSV"""
    df = pd.read_csv(csv_file, encoding='utf-8')
    
    # Get column names
    if 'Input sentence' in df.columns and 'Output sentence' in df.columns:
        input_col = 'Input sentence'
        output_col = 'Output sentence'
    elif 'input' in df.columns and 'output' in df.columns:
        input_col = 'input'
        output_col = 'output'
    else:
        input_col = df.columns[0]
        output_col = df.columns[1]
    
    print(f"📄 File: {csv_file}")
    print(f"📊 Total sentences: {len(df)}")
    print(f"📝 Input column: '{input_col}'")
    print(f"📝 Output column: '{output_col}'")
    
    # Get inputs and outputs
    inputs = df[input_col].astype(str).tolist()
    outputs = df[output_col].astype(str).tolist()
    
    # Calculate GLEU score using the same function
    gleu_score = gleu_proxy(inputs, outputs)
    
    print(f"\n🎯 GLEU Score (Input → Output): {gleu_score:.2f}")
    
    # Calculate some additional statistics
    exact_matches = sum(1 for inp, out in zip(inputs, outputs) if inp.strip() == out.strip())
    different_count = len(df) - exact_matches
    
    print(f"📈 Exact matches: {exact_matches}/{len(df)} ({exact_matches/len(df)*100:.1f}%)")
    print(f"🔄 Different pairs: {different_count}/{len(df)} ({different_count/len(df)*100:.1f}%)")
    
    # Show some examples of different pairs
    print(f"\n📋 Sample corrections from {csv_file}:")
    different_pairs = [(inp, out) for inp, out in zip(inputs, outputs) if inp.strip() != out.strip()]
    for i, (inp, out) in enumerate(different_pairs[:5], 1):
        print(f"{i}.")
        print(f"  Input:  {inp[:100]}")
        print(f"  Output: {out[:100]}")
        print()
    
    return gleu_score

# Calculate GLEU score from dev.csv
print("🔍 Calculating GLEU score directly from dev.csv file:")
dev_gleu = calculate_gleu_from_csv('dev.csv')

🔍 Calculating GLEU score directly from dev.csv file:
📄 File: dev.csv
📊 Total sentences: 101
📝 Input column: 'Input sentence'
📝 Output column: 'Output sentence'

🎯 GLEU Score (Input → Output): 90.23
📈 Exact matches: 25/101 (24.8%)
🔄 Different pairs: 76/101 (75.2%)

📋 Sample corrections from dev.csv:
1.
  Input:  আমার তো মনে হয় মহাকবিরা কিছুতেই সহ্য করতে পারেননি হেলেনের মতো মানসকন্যা একটিমাত্র রাজার রানী হয়ে ধ
  Output: আমার তো মনে হয় মহাকবিরা কিছুতেই সহ্য করতে পারেননি হেলেনের মতো মানসকন্যা একটিমাত্র রাজার রানী হয়ে ধ

2.
  Input:  নারী জাতিকে অবশ্য তুমি সম্পত্তি বলে চিন্তা করছো, গোবিন্দ; তাদের মধ্যে কোহিনুর যারা তাদের জন্যে নাদির
  Output: নারী জাতিকে অবশ্য তুমি সম্পত্তি বলে চিন্তা করছো, গোবিন্দ; তাদের মধ্যে কোহিনুর যারা তাদের জন্যে নাদির

3.
  Input:  না, ঠিক দা নয়।
  Output: না, ঠিক তা নয়।

4.
  Input:  ওই রূপ এবং ওই রুচির মূল্য কী করখ দেওয়া যায় তাই ভাবছি।
  Output: ওই রূপ এবং ওই রুচির মূল্য কী করে দেওয়া যায় তাই ভাবছি।

5.
  Input:  একটি পুরুষ কতটুকু মূল্য দিতে পারঊ?
  Output

In [19]:
# Quick summary of dev.csv GLEU score
print("=" * 50)
print("📊 SUMMARY: GLEU Score from dev.csv")
print("=" * 50)
print(f"🎯 GLEU Score (Input → Output): {dev_gleu:.2f}")
print("=" * 50)

📊 SUMMARY: GLEU Score from dev.csv
🎯 GLEU Score (Input → Output): 85.47


In [29]:
# 📊 Check Actual Dataset Information
import pandas as pd
import os

# Check the Telugu dataset file
telugu_file = 'D:/padhai/gaurav nlp/IndicGEC2025/telugu_dataset_10000.csv'
if os.path.exists(telugu_file):
    df = pd.read_csv(telugu_file, encoding='utf-8')
    print(f"📄 Telugu Dataset: {telugu_file}")
    print(f"📊 Total samples: {len(df):,}")
    print(f"📝 Columns: {list(df.columns)}")
    print(f"📈 First few rows:")
    print(df.head(3))
    print("\n" + "="*50)
else:
    print(f"❌ File not found: {telugu_file}")

# Check current CONFIG file paths
print(f"🔧 Current CONFIG settings:")
print(f"   - Train file: {CONFIG['train_file']}")
print(f"   - Dev file: {CONFIG['dev_file']}")
print(f"   - Output dir: {CONFIG['output_dir']}")

# Check if Bangla files exist
bangla_train = CONFIG['train_file']
bangla_dev = CONFIG['dev_file']

print(f"\n📋 File existence check:")
print(f"   - Bangla train exists: {os.path.exists(bangla_train)}")
print(f"   - Bangla dev exists: {os.path.exists(bangla_dev)}")
print(f"   - Telugu dataset exists: {os.path.exists(telugu_file)}")

# Show current training dataset size if available
try:
    if 'train_df' in locals():
        print(f"\n🎯 Currently loaded dataset:")
        print(f"   - Training samples: {len(train_df):,}")
        print(f"   - Dev samples: {len(dev_df):,}")
        print(f"   - Total: {len(train_df) + len(dev_df):,}")
except:
    print(f"\n❓ No dataset currently loaded in memory")

📄 Telugu Dataset: D:/padhai/gaurav nlp/IndicGEC2025/telugu_dataset_10000.csv
📊 Total samples: 10,000
📝 Columns: ['Input', 'Output']
📈 First few rows:
                                               Input  \
0  భారతదేశం ప్రపంచంలోనే అతి పెద్ద ప్రజాస్వామ్య దేశం.   
1  మనిషి యొక్క స్వార్థం కోసం ప్రకృతిని కాలుష్యం చ...   
2  నేను ప్రస్తుతం ఐఐటి కాన్పూర్ లో పీహెచ్డీ చదువు...   

                                              Output  
0  భారతదేశం ప్రపంచంలోనే అతి పెద్ద ప్రజాస్వామ్య దేశం.  
1  మనిషి యొక్క స్వార్థం కోసం ప్రకృతిని కాలుష్యం చ...  
2  నేను ప్రస్తుతం ఐఐటి కాన్పూర్ లో పీహెచ్డీ చదువు...  

🔧 Current CONFIG settings:
   - Train file: D:/padhai/gaurav nlp/IndicGEC2025/Bangla/train.csv
   - Dev file: D:/padhai/gaurav nlp/IndicGEC2025/Bangla/dev.csv
   - Output dir: ./mt5-bangla-gec-model-fixed

📋 File existence check:
   - Bangla train exists: True
   - Bangla dev exists: False
   - Telugu dataset exists: True

🎯 Currently loaded dataset:
   - Training samples: 475
   - Dev samples: 53
   

In [30]:
# 🔧 Update CONFIG to use Telugu Dataset
# Since you're working with the Telugu dataset, let's update the configuration

UPDATED_CONFIG = CONFIG.copy()
UPDATED_CONFIG.update({
    'train_file': 'D:/padhai/gaurav nlp/IndicGEC2025/telugu_dataset_10000.csv',
    'dev_file': 'D:/padhai/gaurav nlp/IndicGEC2025/Telugu/dev.csv',  # Keep Telugu dev file
    'output_dir': './mt5-telugu-gec-model-fixed',  # Update output directory
})

print("🔄 Updated Configuration for Telugu Dataset:")
print(f"   - Train file: {UPDATED_CONFIG['train_file']}")
print(f"   - Dev file: {UPDATED_CONFIG['dev_file']}")
print(f"   - Output dir: {UPDATED_CONFIG['output_dir']}")

# Check if Telugu dev file exists
telugu_dev = UPDATED_CONFIG['dev_file']
if os.path.exists(telugu_dev):
    dev_df_check = pd.read_csv(telugu_dev, encoding='utf-8')
    print(f"\n✅ Telugu dev file found: {len(dev_df_check)} samples")
else:
    print(f"\n⚠️ Telugu dev file not found: {telugu_dev}")
    print("Will split from training data instead")

# If you want to use this configuration, uncomment the line below:
# CONFIG = UPDATED_CONFIG

print(f"\n📊 Training Progress Summary:")
print(f"   - Your dataset has ~10,000 samples (much larger than I initially thought!)")
print(f"   - Current training at epoch 7.47/35 with GLEU 94.61")
print(f"   - This explains the longer training time and excellent performance")
print(f"   - With 10K samples, the model has plenty of data to learn from")

🔄 Updated Configuration for Telugu Dataset:
   - Train file: D:/padhai/gaurav nlp/IndicGEC2025/telugu_dataset_10000.csv
   - Dev file: D:/padhai/gaurav nlp/IndicGEC2025/Telugu/dev.csv
   - Output dir: ./mt5-telugu-gec-model-fixed

✅ Telugu dev file found: 100 samples

📊 Training Progress Summary:
   - Your dataset has ~10,000 samples (much larger than I initially thought!)
   - Current training at epoch 7.47/35 with GLEU 94.61
   - This explains the longer training time and excellent performance
   - With 10K samples, the model has plenty of data to learn from
